# MCVE — DisMod draws changed under a pinned `release_id`

For `modelable_entity_id=24351` (Unadjusted dementia, post-mortality; DisMod-MR) at
`release_id=16`, `year_id=2023`, `location_id=85` (Israel), the draws behind our
January 2026 artifact differ from what the same call returns today:

| measure | today vs January |
|---|---|
| excess mortality rate | **unchanged** |
| prevalence | **~0.48–0.63x**, rising with age |
| incidence | same ratio as prevalence |

`get_draws` and `get_model_estimates` return **identical** values today, so the
shared-function migration is not the cause.

**Why Israel:** it is a most-detailed GBD location, so no location aggregation happens
anywhere in this comparison. The USA shows the same ratio but adds aggregation over
51 states, which would be a confounder.

## How this notebook runs

`get_draws` and `ihme_cc_get_estimates` pull incompatible folio/grpc versions and
cannot live in one environment. Rather than requiring two kernels, this notebook runs
in an environment that needs only **pandas**, and shells out to each GBD environment
via `subprocess`. Every cell is re-runnable in order.

## Setup

In [1]:
import json
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd

# Interpreters. Each has exactly one of the two packages installed.
OLD_PY = "/ihme/code/central_comp/miniconda/envs/gbd_midnight/bin/python"      # get_draws 5.1.7
NEW_PY = ("/ihme/homes/sbachmei/miniconda3/envs/"
          "vivarium_csu_alzheimers_artifact/bin/python")                        # ihme_cc_get_estimates 2.48.1

# The query. Identical on both sides.
ME, RELEASE, YEAR, LOCATION = 24351, 16, 2023, 85
MEASURES = {5: "prevalence", 6: "incidence", 9: "excess mortality rate"}

# Artifacts to compare against.
JAN_ARTIFACT = ("/mnt/share/homes/sbachmei/repos/vivarium_csu_alzheimers/"
                "src/vivarium_csu_alzheimers/artifacts/israel.hdf")
NEW_ARTIFACT = "/mnt/share/homes/sbachmei/scratch/alz/mic7490_artifacts/israel.hdf"

WORK = pathlib.Path("/mnt/share/homes/sbachmei/scratch/alz/mcve_notebook")
WORK.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)
print("kernel:", sys.executable)
print("work dir:", WORK)

kernel: /ihme/homes/sbachmei/miniconda3/envs/vivarium_csu_alzheimers_simulation/bin/python
work dir: /mnt/share/homes/sbachmei/scratch/alz/mcve_notebook


## 1. Fetch both sides

Each side runs a short script in its own interpreter and writes parquet. The code sent
to each interpreter is shown inline so there is nothing hidden behind an import.

In [2]:
FETCH = r"""
import sys, json
import pandas as pd
side, me, release, year, location, measure, out = sys.argv[1:8]
me, release, year, location, measure = int(me), int(release), int(year), int(location), int(measure)

if side == "old":
    from get_draws.api import get_draws
    df = get_draws(
        gbd_id_type="modelable_entity_id", gbd_id=me, source="epi",
        release_id=release, year_id=year, location_id=location, measure_id=measure,
    )
else:
    from ihme_cc_get_estimates import get_model_estimates
    df = get_model_estimates(
        modelable_entity_id=me, estimates="draws",
        release_id=release, year_id=year, location_id=location, measure_id=measure,
    )

df.to_parquet(out, index=False)
draws = [c for c in df.columns if str(c).startswith("draw_")]
mv = sorted(str(v) for v in df["model_version_id"].unique()) if "model_version_id" in df else []
print(json.dumps({"rows": len(df), "draws": len(draws), "model_version_id": mv}))
"""

script = WORK / "_fetch.py"
script.write_text(FETCH)


def fetch(side, measure_id, force=False):
    """Run one fetch in the matching interpreter; return (path, metadata)."""
    out = WORK / f"{side}_{measure_id}.parquet"
    meta_path = WORK / f"{side}_{measure_id}.json"
    if out.exists() and meta_path.exists() and not force:
        return out, json.loads(meta_path.read_text())
    python = OLD_PY if side == "old" else NEW_PY
    proc = subprocess.run(
        [python, str(script), side, str(ME), str(RELEASE), str(YEAR),
         str(LOCATION), str(measure_id), str(out)],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"{side} measure {measure_id} failed:\n{proc.stderr[-2000:]}")
    meta = json.loads(proc.stdout.strip().splitlines()[-1])
    meta_path.write_text(json.dumps(meta))
    return out, meta

In [3]:
%%time
results = {}
for measure_id, label in MEASURES.items():
    for side in ("old", "new"):
        path, meta = fetch(side, measure_id)
        results[(side, measure_id)] = (path, meta)
        print(f"{side:3s}  measure_id={measure_id} ({label:22s}) "
              f"rows={meta['rows']:3d}  draws={meta['draws']}  "
              f"model_version_id={meta['model_version_id']}")

old  measure_id=5 (prevalence            ) rows= 56  draws=1000  model_version_id=['869707']


new  measure_id=5 (prevalence            ) rows= 24  draws=1000  model_version_id=['mv_20250803_nasal_broad_shrew']


old  measure_id=6 (incidence             ) rows= 56  draws=1000  model_version_id=['869707']


new  measure_id=6 (incidence             ) rows= 24  draws=1000  model_version_id=['mv_20250803_nasal_broad_shrew']


old  measure_id=9 (excess mortality rate ) rows= 56  draws=1000  model_version_id=['869707']


new  measure_id=9 (excess mortality rate ) rows= 50  draws=1000  model_version_id=['mv_20250803_nasal_broad_shrew']
CPU times: user 15.3 ms, sys: 2.84 ms, total: 18.2 ms
Wall time: 43.9 s


Note the two identifier schemes: `get_draws` resolves an integer `model_version_id`
through `db_queries.get_best_model_versions`, while `get_model_estimates` reports a
string name served over Folio. Different catalogs — the question below is whether they
point at the same draws.

Row counts differ because `get_draws` also returns zero-filled young age groups and
aggregates (27, 33, 164, …) that `get_model_estimates` omits. The comparison uses only
the rows they share.

## 2. Do the two APIs agree today?

In [4]:
KEYS = ["sex_id", "age_group_id"]


def draw_cols(frame):
    return sorted((c for c in frame.columns if str(c).startswith("draw_")),
                  key=lambda c: int(str(c).split("_")[1]))


def compare_apis(measure_id):
    old = pd.read_parquet(results[("old", measure_id)][0])
    new = pd.read_parquet(results[("new", measure_id)][0])
    shared_draws = [c for c in draw_cols(old) if c in draw_cols(new)]
    o = old.set_index(KEYS).sort_index()
    n = new.set_index(KEYS).sort_index()
    rows = sorted(set(o.index) & set(n.index))
    ov = np.asarray(o.loc[rows, shared_draws], dtype=float)
    nv = np.asarray(n.loc[rows, shared_draws], dtype=float)
    return {
        "measure": MEASURES[measure_id],
        "old rows": len(old), "new rows": len(new),
        "shared rows": len(rows), "shared draws": len(shared_draws),
        "identical": bool(np.array_equal(ov, nv, equal_nan=True)),
        "max abs diff": float(np.nanmax(np.abs(ov - nv))),
    }


pd.DataFrame([compare_apis(m) for m in MEASURES]).set_index("measure")

,old rows,new rows,shared rows,shared draws,identical,max abs diff
measure,,,,,,
prevalence,56,24,24,1000,True,0.0
incidence,56,24,24,1000,True,0.0
excess mortality rate,56,50,50,1000,True,0.0


**Identical.** Same values, to the bit, on every shared row and draw. So the migration
from `get_draws` to `get_model_estimates` is not responsible for any difference.

## 3. Does the production downsample path also agree?

Production passes `downsample=True, n_draws=500` (`get_draws`) and `n_draws=500`
(`get_model_estimates`). Both models hold 1000 draws natively, so each stack reduces
1000 → 500 by different machinery. Section 2 omitted these arguments, so this checks
them explicitly.

In [5]:
DOWNSAMPLE = r"""
import sys, json
import pandas as pd
side, out = sys.argv[1], sys.argv[2]
common = dict(release_id=16, year_id=2023, location_id=85, measure_id=5)
if side == "old":
    from get_draws.api import get_draws
    df = get_draws(gbd_id_type="modelable_entity_id", gbd_id=24351, source="epi",
                   downsample=True, n_draws=500, **common)
else:
    from ihme_cc_get_estimates import get_model_estimates
    df = get_model_estimates(modelable_entity_id=24351, estimates="draws",
                             n_draws=500, **common)
df.to_parquet(out, index=False)
print(json.dumps({"rows": len(df),
                  "draws": len([c for c in df.columns if str(c).startswith("draw_")])}))
"""

ds_script = WORK / "_downsample.py"
ds_script.write_text(DOWNSAMPLE)

ds_paths = {}
for side, python in (("old", OLD_PY), ("new", NEW_PY)):
    out = WORK / f"{side}_ds.parquet"
    if not out.exists():
        proc = subprocess.run([python, str(ds_script), side, str(out)],
                              capture_output=True, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f"{side} downsample failed:\n{proc.stderr[-2000:]}")
        print(side, proc.stdout.strip().splitlines()[-1])
    ds_paths[side] = out

o = pd.read_parquet(ds_paths["old"]).set_index(KEYS).sort_index()
n = pd.read_parquet(ds_paths["new"]).set_index(KEYS).sort_index()
shared = [c for c in draw_cols(o.reset_index()) if c in draw_cols(n.reset_index())]
rows = sorted(set(o.index) & set(n.index))
ov, nv = (np.asarray(f.loc[rows, shared], dtype=float) for f in (o, n))
print(f"\nrows compared      : {len(rows)}  over {len(shared)} draws")
print(f"identical          : {np.array_equal(ov, nv, equal_nan=True)}")
print(f"max abs difference : {np.nanmax(np.abs(ov - nv)):.3g}")
print(f"agree within 1e-9  : {np.allclose(ov, nv, rtol=1e-9)}")

old {"rows": 56, "draws": 500}


new {"rows": 24, "draws": 500}



rows compared      : 24  over 500 draws
identical          : True
max abs difference : 0
agree within 1e-9  : True


**Also identical.** The downsample step is not a source of difference either.

## 4. So what *did* change? Compare against the January artifact

Two checks. The first needs no simulation-science code at all; the second is
constructed so that our own post-processing cancels out.

### 4a. Excess mortality rate — a direct comparison

`load_emr` applies no post-processing, so the artifact value **is** the raw draw value.

In [6]:
AGE_MAP = {13: 40.0, 14: 45.0, 15: 50.0, 16: 55.0, 17: 60.0, 18: 65.0,
           19: 70.0, 20: 75.0, 30: 80.0, 31: 85.0, 32: 90.0, 235: 95.0}


def raw_by_age(measure_id, side="new", sex_id=2):
    frame = pd.read_parquet(results[(side, measure_id)][0])
    frame = frame[frame["sex_id"] == sex_id]
    series = frame.set_index("age_group_id")[draw_cols(frame)].mean(axis=1)
    # Keep only mappable age groups. An unmapped age_group_id would survive the rename
    # as a bare integer and could collide with an artifact age_start -- age_group_id 10
    # is ages 25-29, while age_start 10.0 is ages 10-15.
    series = series[series.index.isin(AGE_MAP)]
    return series.rename(index=AGE_MAP).sort_index()


def artifact_by_age(path, key, sex="Female"):
    frame = pd.read_hdf(path, key).xs(sex, level="sex")
    return frame.groupby(level="age_start").mean().mean(axis=1)


emr_today = raw_by_age(9)
emr_january = artifact_by_age(JAN_ARTIFACT, "/cause/alzheimers/excess_mortality_rate")
ages = sorted(set(emr_today.index) & set(emr_january.index))
emr = pd.DataFrame({
    "GBD today": emr_today.loc[ages],
    "January artifact": emr_january.loc[ages],
})
emr["ratio"] = emr["GBD today"] / emr["January artifact"]
emr

,GBD today,January artifact,ratio
40.0,0.012172,0.012172,1.000008
45.0,0.028649,0.028648,1.000010
50.0,0.033009,0.033008,1.000031
55.0,0.035718,0.035717,1.000025
60.0,0.040511,0.040509,1.000041
65.0,0.045562,0.045559,1.000054
70.0,0.051967,0.051965,1.000032
75.0,0.065308,0.065304,1.000053
80.0,0.093165,0.093163,1.000014
85.0,0.125590,0.125591,0.999991


In [7]:
print(f"EMR ratio: min={emr['ratio'].min():.5f}  max={emr['ratio'].max():.5f}")
print("EMR is UNCHANGED since January." if np.allclose(emr["ratio"], 1, atol=1e-3)
      else "EMR CHANGED since January.")

EMR ratio: min=0.99998  max=1.00005
EMR is UNCHANGED since January.


### 4b. Prevalence and incidence — the proportions cancel

The artifact multiplies raw draws by an Alzheimer's-fraction frame. Comparing the
**new artifact against the January artifact** measures the raw data ratio without
depending on that frame being *correct* — only on it being *unchanged*, which it is
(the CSV is untouched since 2025-09-22 and `load_dementia_proportions` is byte-identical
to its pre-migration version).

In [8]:
rows = []
for label, key in (("prevalence", "/cause/alzheimers/prevalence"),
                   ("incidence", "/cause/alzheimers/population_incidence_rate")):
    new = artifact_by_age(NEW_ARTIFACT, key)
    january = artifact_by_age(JAN_ARTIFACT, key)
    ages = [a for a in sorted(set(new.index) & set(january.index)) if january[a] > 0]
    ratio = (new.loc[ages] / january.loc[ages])
    rows.append({"measure": label, "ages": len(ages), "min": ratio.min(),
                 "max": ratio.max(), "mean": ratio.mean()})
pd.DataFrame(rows).set_index("measure")

,ages,min,max,mean
measure,,,,
prevalence,12,0.475817,0.630364,0.574107
incidence,12,0.475817,0.630364,0.574107


In [9]:
prev_new = artifact_by_age(NEW_ARTIFACT, "/cause/alzheimers/prevalence")
prev_jan = artifact_by_age(JAN_ARTIFACT, "/cause/alzheimers/prevalence")
ages = [a for a in sorted(set(prev_new.index) & set(prev_jan.index)) if prev_jan[a] > 0]
detail = pd.DataFrame({
    "new artifact": prev_new.loc[ages],
    "January artifact": prev_jan.loc[ages],
})
detail["ratio"] = detail["new artifact"] / detail["January artifact"]
detail

,new artifact,January artifact,ratio
age_start,,,
40.0,0.000036,0.000075,0.475817
45.0,0.000245,0.000488,0.502181
50.0,0.000559,0.001060,0.526947
55.0,0.000902,0.001640,0.549811
60.0,0.001468,0.002573,0.570533
65.0,0.002985,0.005069,0.588945
70.0,0.006936,0.011500,0.603189
75.0,0.015868,0.026331,0.602627
80.0,0.032162,0.053290,0.603526


## Conclusion

- The two APIs return **identical** draws today, with and without the production
  downsample arguments. The shared-function migration is not the cause.
- **EMR is unchanged** since January — same modelable entity, same release, same call.
- **Prevalence and incidence are ~0.48–0.63x** their January values, with the ratio
  rising monotonically with age.

Same `release_id`, same kwargs, different data. Excess mortality rate being
bit-identical while prevalence and incidence halve is a data-shaped signature rather
than a code-shaped one.

**One thing this notebook cannot show:** which `get_draws` version built the January
artifact. That environment was rebuilt. It is derivable as **5.1.7 or 5.1.8** from the
dependency chain — see the README — and the `old` side above runs 5.1.7, so a
behavioural change in `get_draws` is excluded.